# finance / household / networth

**Intent**
- Outputs `finance_household_networth_current_dollars`, net worth at the latest month end.
- Outputs `finance_household_networth_savings_rate_pct`, the savings rate over the period.
- Gives constants, with the starting wealth the portfolio models need.
- Closes the loop between the plan's forecasts and what happens.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is twelve synthetic months of net worth, income and spending.
- It is synthetic until the `bank` table exists.
- The `Balances` tab of the `Bank` workbook holds a reading per account over time from 2021-05.

**Status**
- The real series is not loaded and nothing is released.
- It carries no evaluation, so the golden replay and the Fit cell assertions are the gate.

**Future**
- Add a `bank` table to `wrangle`.
- Build the net worth series, reconcile it to income less spending, and separate saving from market return.
- Add the tracking fan against the base, low and high paths once `portfolio/return` is real.
- Add property, super and other assets with dated values, and liabilities with their rates.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/household/networth"
MODEL_NAME = "finance_household_networth"
TARGET = {"kind": "local"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "household_networth_training_2026-11-05_2026-11-05"
GOLDEN = "household_networth_golden"
INPUT = INPUTS[0]

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {
        "month": list(range(12)),
        "net_worth_dollars": [1000000.0 + 5000.0 * month for month in range(12)],
        "income_dollars": [12000.0] * 12,
        "spending_dollars": [9000.0] * 12,
    },
)
print("training", training_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
income = float(training_pl["income_dollars"].sum())
spending = float(training_pl["spending_dollars"].sum())
CONSTANTS = dict(zip(OUTPUTS, [round(float(training_pl["net_worth_dollars"][-1]), 0), round((income - spending) / income * 100.0, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [1055000.0, 25.0], "the summary did not give the latest net worth and the savings rate"

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench.model import package_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic monthly net worth, income and spending, training split",
        [training_path, golden_path],
        "latest net worth and the savings rate over the period",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped loader")